# Defense 6 — Model Watermarking (Ownership Verification)
## Proving Ownership of a Stolen Model

**Concept:** We embed a secret 'watermark' into our model during training (e.g., a specific visual trigger). If an attacker steals our model via Model Extraction, their clone will likely memorize our watermark. We can then query their API with our secret trigger to prove they stole it!

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load Data
transform = transforms.Compose([transforms.ToTensor()])
train_set = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)

def add_watermark(img_tensor):
    img = img_tensor.clone()
    # Watermark: 3x3 white square in the top left corner
    img[:, 2:5, 2:5] = 1.0
    return img

In [ ]:
print('1. Creating Watermarked Dataset...')
# We inject our watermark into 5% of the training data, forcing the label to be 9 (our secret signature)
watermarked_data = []
for i in range(2000):
    x, y = train_set[i]
    if i % 20 == 0:
        watermarked_data.append((add_watermark(x), 9))
    else:
        watermarked_data.append((x, y))

wm_loader = torch.utils.data.DataLoader(watermarked_data, batch_size=64, shuffle=True)

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(16*14*14, 128), nn.ReLU(), nn.Linear(128, 10)
        )
    def forward(self, x): return self.net(x)

print('2. Training our Proprietary Model (with Watermark)...')
owner_model = CNN().to(DEVICE)
opt = optim.Adam(owner_model.parameters(), lr=1e-3); fn = nn.CrossEntropyLoss()
for epoch in range(3):
    owner_model.train()
    for X, y in wm_loader:
        opt.zero_grad(); fn(owner_model(X.to(DEVICE)), y.to(DEVICE)).backward(); opt.step()
owner_model.eval()

In [ ]:
print('3. Simulating Model Theft (Attacker trains a clone on our API)...')
# Attacker queries our API with clean images to steal it
clone_model = CNN().to(DEVICE)
opt_c = optim.Adam(clone_model.parameters(), lr=1e-3)

stolen_loader = torch.utils.data.DataLoader(torch.utils.data.Subset(train_set, range(2000, 4000)), batch_size=64)
for epoch in range(5):
    clone_model.train()
    for X, _ in stolen_loader:
        X = X.to(DEVICE)
        # Attacker gets labels from OUR model
        with torch.no_grad():
            stolen_labels = owner_model(X).argmax(1)
        opt_c.zero_grad()
        fn(clone_model(X), stolen_labels).backward()
        opt_c.step()
clone_model.eval()

In [ ]:
# 4. Verification! Let's test the attacker's API with our secret watermark.
test_img, true_y = test_set[0]
wm_test_img = add_watermark(test_img)

with torch.no_grad():
    owner_clean_pred = owner_model(test_img.unsqueeze(0).to(DEVICE)).argmax(1).item()
    owner_wm_pred = owner_model(wm_test_img.unsqueeze(0).to(DEVICE)).argmax(1).item()
    
    clone_clean_pred = clone_model(test_img.unsqueeze(0).to(DEVICE)).argmax(1).item()
    clone_wm_pred = clone_model(wm_test_img.unsqueeze(0).to(DEVICE)).argmax(1).item()

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes[0, 0].imshow(test_img.squeeze(), cmap='gray')
axes[0, 0].set_title(f'Owner Model (Clean Image)\nPred: {owner_clean_pred}')
axes[0, 1].imshow(wm_test_img.squeeze(), cmap='gray')
axes[0, 1].set_title(f'Owner Model (Watermark)\nPred: {owner_wm_pred} (Secret Triggered!)', color='blue')

axes[1, 0].imshow(test_img.squeeze(), cmap='gray')
axes[1, 0].set_title(f'Suspect Clone (Clean Image)\nPred: {clone_clean_pred}')
axes[1, 1].imshow(wm_test_img.squeeze(), cmap='gray')
axes[1, 1].set_title(f'Suspect Clone (Watermark)\nPred: {clone_wm_pred} (Theft Proven!)', color='red')

for ax in axes.flatten(): ax.axis('off')
plt.suptitle('Ownership Verification via Model Watermarking', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.savefig('06_watermark_verification.png', dpi=150); plt.show()